In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class SymbolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        # Normalize (VERY IMPORTANT)
        self.X = self.X / (torch.norm(self.X, dim=1, keepdim=True) + 1e-8)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SymbolClassifier(nn.Module):
    def __init__(self,mean,std):
        super().__init__()
        # Register as buffers (saved with model, not trainable)
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
        self.register_buffer("std",  torch.tensor(std,  dtype=torch.float32))
        
        self.net = nn.Sequential(
            nn.Linear(16, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512)
        )

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.net(x)  # logits

In [2]:
# Example placeholders
sf = 9
input = 16
output = 512

folder_path = "classifier_dataset_sf{}_{}_{}".format(sf, input, output)

X = np.load(f"{folder_path}/X.npy")   # shape (30720, 16)
y = np.load(f"{folder_path}/y.npy")   # shape (30720,)
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8  # avoid divide by zero
print(X.shape, y.shape)

print(y)
dataset = SymbolDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SymbolClassifier(X_mean, X_std).to(device)
criterion = nn.CrossEntropyLoss()   # Softmax included
optimizer = optim.Adam(model.parameters(), lr=1e-4)

(30720, 16) (30720,)
[  0   1   2 ... 509 510 511]


In [ ]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)              # (batch, 512)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    acc = correct / total * 100

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

Epoch [1/100] Loss: 5.9526 | Accuracy: 2.23%
Epoch [2/100] Loss: 4.4948 | Accuracy: 5.35%
Epoch [3/100] Loss: 3.9074 | Accuracy: 8.01%
Epoch [4/100] Loss: 3.6841 | Accuracy: 9.63%
Epoch [5/100] Loss: 3.5379 | Accuracy: 10.49%
Epoch [6/100] Loss: 3.4298 | Accuracy: 11.38%
Epoch [7/100] Loss: 3.3415 | Accuracy: 11.94%
Epoch [8/100] Loss: 3.2682 | Accuracy: 13.12%
Epoch [9/100] Loss: 3.2061 | Accuracy: 13.18%
Epoch [10/100] Loss: 3.1509 | Accuracy: 13.88%
Epoch [11/100] Loss: 3.1028 | Accuracy: 14.38%
Epoch [12/100] Loss: 3.0592 | Accuracy: 15.03%
Epoch [13/100] Loss: 3.0197 | Accuracy: 15.23%
Epoch [14/100] Loss: 2.9865 | Accuracy: 15.61%
Epoch [15/100] Loss: 2.9517 | Accuracy: 15.90%
Epoch [16/100] Loss: 2.9220 | Accuracy: 16.71%
Epoch [17/100] Loss: 2.8935 | Accuracy: 17.28%
Epoch [18/100] Loss: 2.8670 | Accuracy: 17.62%
Epoch [19/100] Loss: 2.8427 | Accuracy: 18.06%
Epoch [20/100] Loss: 2.8199 | Accuracy: 18.08%
Epoch [21/100] Loss: 2.7973 | Accuracy: 18.80%
Epoch [22/100] Loss: 2.777

In [ ]:


def evaluate(model, dataloader):
    model.eval()
    correct1 = 0
    correct5 = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            _, top5 = torch.topk(logits, k=5, dim=1)
            preds = torch.argmax(logits, dim=1)

            correct1 += (preds == y_batch).sum().item()
            correct5 += (top5 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total += y_batch.size(0)

    print(f"Top-1 Accuracy: {100*correct1/total:.2f}%")
    print(f"Top-5 Accuracy: {100*correct5/total:.2f}%")

# Save
torch.save(model.state_dict(), "symbol_classifier.pt")

# Load
model.load_state_dict(torch.load("symbol_classifier.pt"))
model.eval()
evaluate(model,dataloader)

Top-1 Accuracy: 14.77%
Top-5 Accuracy: 57.96%
